## Shared helpers for 01_model_log

Not meant to be run alone. Use **%run ./00_shared** from 1a–1e to load these functions into the current notebook (no module import).

In [ ]:
import os
import json
import hashlib

import numpy as np
import pandas as pd
import torch
import mlflow
from mlflow.tracking import MlflowClient


def get_artifact_root(catalog: str, schema: str, volume: str) -> str:
    return f"/Volumes/{catalog}/{schema}/{volume}"


def get_native_xgb_paths(artifact_root: str) -> dict:
    return {"path": os.path.join(artifact_root, "xgb.pkl"), "checksum_key": "xgb.pkl"}


def get_native_sklearn_paths(artifact_root: str) -> dict:
    return {
        "path": os.path.join(artifact_root, "sklearn_model.pkl"),
        "checksum_key": "sklearn_model.pkl",
    }


def get_pyfunc_xgb_paths(artifact_root: str) -> dict:
    return {
        "xgb_model": os.path.join(artifact_root, "pyfunc_xgb.pkl"),
        "preproc": os.path.join(artifact_root, "pyfunc_preproc.json"),
        "postproc": os.path.join(artifact_root, "pyfunc_postproc.json"),
    }


def get_dl_config(artifact_root: str) -> dict:
    return {
        "artifacts_json_path": os.path.join(artifact_root, "artifacts.json"),
        "torch_model_path": os.path.join(artifact_root, "torch_model.pt"),
    }


def get_canonical_input_path(artifact_root: str) -> str:
    return os.path.join(artifact_root, "canonical_input.json")


def get_checksums_path(artifact_root: str) -> str:
    return os.path.join(artifact_root, "checksums.json")


def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def load_expected_checksums(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def verify_checksum(path: str, expected_hex: str) -> None:
    actual = sha256_file(path)
    if actual != expected_hex:
        raise ValueError(f"[Checksum] Mismatch for {path}. expected={expected_hex}, actual={actual}")


def verify_artifacts(checksums_path: str, items: list) -> None:
    """items = [(checksum_key, file_path), ...]. Verifies only keys present in checksums."""
    if not os.path.isfile(checksums_path):
        return
    expected = load_expected_checksums(checksums_path)
    for name, path in items:
        if name in expected:
            verify_checksum(path, expected[name])
    print("Checksum verification passed for specified artifacts.")


def load_canonical_input_json(canonical_input_path: str):
    """Return (canonical_input DataFrame, feature_columns list)."""
    canonical_input = pd.read_json(canonical_input_path, orient="records").astype("float64")
    feature_columns = list(canonical_input.columns)
    if canonical_input.shape[0] < 1:
        raise ValueError(f"Canonical input must have at least one row; got shape={canonical_input.shape}")
    return canonical_input, feature_columns


def load_canonical_input_from_artifacts_json(artifacts_json_path: str):
    """Return (canonical_input DataFrame, feature_columns list)."""
    with open(artifacts_json_path, "r", encoding="utf-8") as f:
        ad = json.load(f)
    ex = ad["example_input"]
    feature_columns = ad.get("feature_columns") or [f"f{i}" for i in range(len(ex[0]))]
    return pd.DataFrame(ex, columns=feature_columns).astype("float64"), feature_columns


def validate_and_infer_signature(canonical_input: pd.DataFrame, y_pred: pd.DataFrame, expected_cols=None):
    if expected_cols is None:
        expected_cols = ["prediction"]
    if not isinstance(y_pred, pd.DataFrame) or list(y_pred.columns) != expected_cols:
        raise ValueError(
            f"Output must be DataFrame with columns {expected_cols}; "
            f"got {type(y_pred)} {list(getattr(y_pred, 'columns', []))}"
        )
    return mlflow.models.infer_signature(canonical_input, y_pred)


def load_torch_artifact(path: str):
    """Load torch_model.pt; return (model, scaler_or_none). Handles full model, bundle {model, scaler, meta}, or state_dict."""
    loaded = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(loaded, dict) and "model" in loaded and "scaler" in loaded:
        return loaded["model"], loaded["scaler"]
    return loaded, None


def register_in_uc(
    client: MlflowClient,
    model_uri: str,
    registered_name: str,
    version: str,
    alias: str = "Champion",
) -> None:
    client.set_model_version_tag(registered_name, version, key="approval", value="approved")
    client.set_registered_model_alias(registered_name, alias=alias, version=version)
    mv = client.get_model_version_by_alias(registered_name, alias)
    assert str(mv.version) == str(version), f"Alias {alias} expected version {version}, got {mv.version}"
    assert client.get_model_version(registered_name, mv.version).tags.get("approval") == "approved"
